In [ ]:
# pip install langchain
# pip install langchain-core
# pip install langchain-community
# pip install langchain-classic
# pip install langchain-huggingface
# pip install langchain-pinecone
# pip install langchain-text-splitters
# pip install pinecone
# pip install ctransformers
# pip install sentence-transformers
# pip install pypdf

SyntaxError: invalid syntax (522252386.py, line 2)

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import RetrievalQA

c:\Users\user\anaconda3\envs\menv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_13788\1515216797.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [3]:
pinecone_api_key = 'pcsk_4azADL_BvSAGCXvzZGCBnkwBVTkni2jtKAwigKVSG8PyZF2Z6Qb3htepMazqYkaahgUtj9'
pinecond_api_env = 'gcp_starter'

In [7]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
def load_pdf(data):
    loader = DirectoryLoader(data,
                             glob="*.pdf",
                             loader_cls = PyPDFLoader)
    documents = loader.load()
    return documents

In [8]:
extract_data = load_pdf("Data/")

In [9]:
def text_split(extract_data):
   text_splitter =  RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
   text_chunks = text_splitter.split_documents(extract_data)

   return text_chunks
    

In [10]:
text_chunks = text_split(extract_data)
print(len(text_chunks))

0


In [11]:
text_chunks

[]

In [12]:
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [13]:
embeddings = download_hugging_face_embeddings()

c:\Users\user\anaconda3\envs\menv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 469.99it/s]


In [14]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [19]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import os

pinecone_api_key = 'pcsk_4azADL_BvSAGCXvzZGCBnkwBVTkni2jtKAwigKVSG8PyZF2Z6Qb3htepMazqYkaahgUtj9'

os.environ["PINECONE_API_KEY"] = pinecone_api_key

pc = Pinecone(api_key=pinecone_api_key)

index_name = "medical-chatbot"

# create index
if index_name not in pc.list_indexes().names():

    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# vector store
docsearch = PineconeVectorStore.from_texts(
    [t.page_content for t in text_chunks],
    embedding=embeddings,
    index_name=index_name
)

In [21]:
TEMMPLATE = """
User the following pieces of information to answer the user questions.
If you don't know the answer, just say I dn't know, don't try to makeup the answer.
Context: {context}
Question: {question}
Only return the helpful anser the below and nothing else.
Helpful answer.
"""

In [24]:
PROMPT = PromptTemplate(
    template=TEMMPLATE,
    input_variables=["context", "question"]
)
chain_type_kwargs = {"prompt": PROMPT}


In [27]:
from langchain_community.llms import CTransformers

llm = CTransformers(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    model_type="llama",
    config={
        "max_new_tokens": 512,
        "temperature": 0.8
    }
)

Fetching 1 files:   0%|          | 0/1 [00:02<?, ?it/s]


GatedRepoError: 401 Client Error. (Request ID: Root=1-6a12040f-476743362534c3f235a444f7;badad5f6-0d8b-4ede-9d4e-584ce8e747da)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [28]:
qa = RetrievalQA.from_chain_type(
    LLm = llm,
    chain_type = "stuff",
    retriever = docsearch.as_retriever(search_kwargs = {'K': 2}),
    return_source_documents = True,
    chain_type_kwargs = chain_type_kwargs
)

NameError: name 'llm' is not defined

In [ ]:
while True:
    user_input = input(f"Input Prompt:")
    result = qa({"query": user_input})
    print("Response: ", result["result"])